# Optimization of Kerosene FT Process via SAC (Soft Actor-Critic) RL

## 1. Import Packages and Check CUDA

In [1]:
import csv  # :) --<
import random
import re
import time
from collections import deque
from pathlib import Path

import gymnasium as gym
import numpy as np
import pandas as pd
import plotly.express as px
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from pythoncom import com_error
from torch.distributions import Normal

if torch.cuda.is_available():
    print("CUDA is available. GPU can be used.")
else:
    print("CUDA is not available. Using CPU.")


CUDA is available. GPU can be used.


## 2. Define SAC Components

In [2]:
class BasicBuffer:
    def __init__(self, max_size):
        self.buffer = deque(maxlen=max_size)

    def push(self, state, action, reward, next_state, done):
        experience = (
            np.asarray(state, dtype=np.float32),
            np.asarray(action, dtype=np.float32),
            np.asarray([reward], dtype=np.float32),
            np.asarray(next_state, dtype=np.float32),
            float(done),
        )
        self.buffer.append(experience)

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = map(np.array, zip(*batch))
        return states, actions, rewards, next_states, dones

    def __len__(self):
        return len(self.buffer)


class SoftQNetwork(nn.Module):
    def __init__(self, num_inputs, num_actions, hidden_size=256, init_w=3e-3):
        super().__init__()
        self.linear1 = nn.Linear(num_inputs + num_actions, hidden_size)
        self.linear2 = nn.Linear(hidden_size, hidden_size)
        self.linear3 = nn.Linear(hidden_size, 1)

        self.linear3.weight.data.uniform_(-init_w, init_w)
        self.linear3.bias.data.uniform_(-init_w, init_w)

    def forward(self, state, action):
        x = torch.cat([state, action], dim=1)
        x = F.relu(self.linear1(x))
        x = F.relu(self.linear2(x))
        return self.linear3(x)


class PolicyNetwork(nn.Module):
    def __init__(self, num_inputs, num_actions, hidden_size=256, init_w=3e-3, log_std_min=-20, log_std_max=2):
        super().__init__()
        self.log_std_min = log_std_min
        self.log_std_max = log_std_max

        self.linear1 = nn.Linear(num_inputs, hidden_size)
        self.linear2 = nn.Linear(hidden_size, hidden_size)
        self.mean_linear = nn.Linear(hidden_size, num_actions)
        self.log_std_linear = nn.Linear(hidden_size, num_actions)

        self.mean_linear.weight.data.uniform_(-init_w, init_w)
        self.mean_linear.bias.data.uniform_(-init_w, init_w)
        self.log_std_linear.weight.data.uniform_(-init_w, init_w)
        self.log_std_linear.bias.data.uniform_(-init_w, init_w)

    def forward(self, state):
        x = F.relu(self.linear1(state))
        x = F.relu(self.linear2(x))
        mean = self.mean_linear(x)
        log_std = torch.clamp(self.log_std_linear(x), self.log_std_min, self.log_std_max)
        return mean, log_std

    def sample(self, state, epsilon=1e-6):
        mean, log_std = self.forward(state)
        std = log_std.exp()
        normal = Normal(mean, std)
        z = normal.rsample()
        action = torch.tanh(z)
        log_pi = normal.log_prob(z) - torch.log(1 - action.pow(2) + epsilon)
        log_pi = log_pi.sum(dim=1, keepdim=True)
        return action, log_pi


class SACAgent:
    def __init__(self, env, gamma, tau, alpha, q_lr, policy_lr, a_lr, buffer_maxlen):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.env = env
        self.gamma = gamma
        self.tau = tau
        self.update_step = 0
        self.delay_step = 1

        obs_dim = env.observation_space.shape[0]
        act_dim = env.action_space.shape[0]
        self.action_low = env.action_space.low
        self.action_high = env.action_space.high

        self.q_net1 = SoftQNetwork(obs_dim, act_dim).to(self.device)
        self.q_net2 = SoftQNetwork(obs_dim, act_dim).to(self.device)
        self.target_q_net1 = SoftQNetwork(obs_dim, act_dim).to(self.device)
        self.target_q_net2 = SoftQNetwork(obs_dim, act_dim).to(self.device)
        self.policy_net = PolicyNetwork(obs_dim, act_dim).to(self.device)

        self.target_q_net1.load_state_dict(self.q_net1.state_dict())
        self.target_q_net2.load_state_dict(self.q_net2.state_dict())

        self.q1_optimizer = optim.Adam(self.q_net1.parameters(), lr=q_lr)
        self.q2_optimizer = optim.Adam(self.q_net2.parameters(), lr=q_lr)
        self.policy_optimizer = optim.Adam(self.policy_net.parameters(), lr=policy_lr)

        self.alpha = float(alpha)
        self.alpha_min = 0.05
        self.alpha_max = 0.50
        self.target_entropy = -float(np.prod(env.action_space.shape))
        initial_log_alpha = np.log(max(self.alpha, 1e-6))
        self.log_alpha = torch.tensor([initial_log_alpha], dtype=torch.float32, device=self.device, requires_grad=True)
        self.alpha_optim = optim.Adam([self.log_alpha], lr=a_lr)

        self.replay_buffer = BasicBuffer(buffer_maxlen)
        self.training_stats = {
            'q1_loss': np.nan,
            'q2_loss': np.nan,
            'policy_loss': np.nan,
            'alpha_loss': np.nan,
            'alpha': float(self.alpha),
        }

    def rescale_action(self, action):
        return action * (self.action_high - self.action_low) / 2.0 + (self.action_high + self.action_low) / 2.0

    def get_action(self, state):
        state_tensor = torch.as_tensor(state, dtype=torch.float32, device=self.device).unsqueeze(0)
        mean, log_std = self.policy_net(state_tensor)
        std = log_std.exp()
        normal = Normal(mean, std)
        z = normal.sample()
        action = torch.tanh(z).detach().cpu().numpy().squeeze(0)
        return self.rescale_action(action).astype(np.float32)

    def update(self, batch_size):
        states, actions, rewards, next_states, dones = self.replay_buffer.sample(batch_size)

        states = torch.as_tensor(states, dtype=torch.float32, device=self.device)
        actions = torch.as_tensor(actions, dtype=torch.float32, device=self.device)
        rewards = torch.as_tensor(rewards, dtype=torch.float32, device=self.device)
        next_states = torch.as_tensor(next_states, dtype=torch.float32, device=self.device)
        dones = torch.as_tensor(dones, dtype=torch.float32, device=self.device).view(-1, 1)

        next_actions, next_log_pi = self.policy_net.sample(next_states)
        next_q1 = self.target_q_net1(next_states, next_actions)
        next_q2 = self.target_q_net2(next_states, next_actions)
        next_q_target = torch.min(next_q1, next_q2) - self.alpha * next_log_pi
        expected_q = rewards + (1 - dones) * self.gamma * next_q_target

        curr_q1 = self.q_net1(states, actions)
        curr_q2 = self.q_net2(states, actions)
        q1_loss = F.mse_loss(curr_q1, expected_q.detach())
        q2_loss = F.mse_loss(curr_q2, expected_q.detach())

        self.q1_optimizer.zero_grad()
        q1_loss.backward()
        self.q1_optimizer.step()

        self.q2_optimizer.zero_grad()
        q2_loss.backward()
        self.q2_optimizer.step()

        policy_loss_value = np.nan
        new_actions, log_pi = self.policy_net.sample(states)
        if self.update_step % self.delay_step == 0:
            min_q = torch.min(self.q_net1(states, new_actions), self.q_net2(states, new_actions))
            policy_loss = (self.alpha * log_pi - min_q).mean()
            policy_loss_value = float(policy_loss.item())

            self.policy_optimizer.zero_grad()
            policy_loss.backward()
            self.policy_optimizer.step()

            for target_param, param in zip(self.target_q_net1.parameters(), self.q_net1.parameters()):
                target_param.data.copy_(self.tau * param.data + (1 - self.tau) * target_param.data)

            for target_param, param in zip(self.target_q_net2.parameters(), self.q_net2.parameters()):
                target_param.data.copy_(self.tau * param.data + (1 - self.tau) * target_param.data)

        alpha_loss = (self.log_alpha * (-log_pi - self.target_entropy).detach()).mean()
        self.alpha_optim.zero_grad()
        alpha_loss.backward()
        self.alpha_optim.step()
        with torch.no_grad():
            self.log_alpha.clamp_(min=np.log(self.alpha_min), max=np.log(self.alpha_max))
        self.alpha = float(self.log_alpha.exp().item())
        self.training_stats = {
            'q1_loss': float(q1_loss.item()),
            'q2_loss': float(q2_loss.item()),
            'policy_loss': policy_loss_value,
            'alpha_loss': float(alpha_loss.item()),
            'alpha': float(self.alpha),
        }
        self.update_step += 1
        return self.training_stats


def mini_batch_train(env, agent, max_episodes, max_steps, batch_size, filename):
    csv_files = {
        'reward': f'report_rewards_{filename}.csv',
        'action': f'report_actions_{filename}.csv',
        'state': f'report_states_{filename}.csv',
        'time': f'report_runTime_{filename}.csv',
    }

    headers = {
        'reward': ['episode', 'total_reward'],
        'action': ['episode', 'action'],
        'state': ['episode', 'last_state'],
        'time': ['episode', 'runTime_sec'],
    }

    for key, path in csv_files.items():
        with open(path, 'w', newline='') as file:
            csv.writer(file).writerow(headers[key])

    episode_rewards = []
    update_metrics_window = deque(maxlen=100)
    best_step_reward = -np.inf
    best_step_action = env.nominal_action.copy()
    best_episode_reward = -np.inf
    best_episode_action = env.nominal_action.copy()
    updates_per_step = 6
    warmup_episodes = min(20, max(10, max_episodes // 20))
    epsilon_start = 0.85
    epsilon_end = 0.05
    epsilon_decay_episodes = max(warmup_episodes + 1, int(max_episodes * 0.70))
    action_span = np.asarray(env.high_act - env.low_act, dtype=np.float32)
    min_noise = np.maximum(0.004 * action_span, np.array([0.05, 0.002, 0.002], dtype=np.float32))
    warmup_noise = np.maximum(0.28 * action_span, np.array([4.5, 0.180, 0.180], dtype=np.float32))
    explore_noise_start = np.maximum(0.18 * action_span, np.array([3.0, 0.120, 0.120], dtype=np.float32))
    explore_noise_end = np.maximum(0.03 * action_span, np.array([0.50, 0.020, 0.020], dtype=np.float32))
    policy_noise_start = np.maximum(0.10 * action_span, np.array([1.50, 0.050, 0.050], dtype=np.float32))
    policy_noise_end = np.maximum(0.008 * action_span, np.array([0.10, 0.004, 0.004], dtype=np.float32))
    probe_noise_scale = np.maximum(0.30 * action_span, np.array([5.0, 0.180, 0.180], dtype=np.float32))
    stability_replay_start = 0.15
    stability_replay_end = 0.85
    full_range_prob_start = 0.55
    full_range_prob_end = 0.10
    probe_episode_every = 6
    max_total_com_errors = 12
    total_com_errors = 0
    consecutive_com_errors = 0

    for episode in range(max_episodes):
        start_time = time.time()
        state, _ = env.reset()
        episode_reward = 0.0
        episode_best_reward = -np.inf
        episode_best_action = None
        action = env.nominal_action.copy()
        last_applied_action = env.nominal_action.copy()
        next_state = state

        progress = episode / max(1, max_episodes - 1)
        epsilon_progress = min(max(episode - warmup_episodes, 0) / max(1, epsilon_decay_episodes - warmup_episodes), 1.0)
        epsilon = 1.0 if episode < warmup_episodes else epsilon_start + (epsilon_end - epsilon_start) * epsilon_progress
        explore_noise_scale = warmup_noise.copy() if episode < warmup_episodes else explore_noise_start + epsilon_progress * (explore_noise_end - explore_noise_start)
        policy_noise_scale = policy_noise_start + epsilon_progress * (policy_noise_end - policy_noise_start)
        stability_replay_prob = stability_replay_start + progress * (stability_replay_end - stability_replay_start)
        full_range_prob = np.clip(full_range_prob_start + progress * (full_range_prob_end - full_range_prob_start), 0.08, 0.60)

        is_probe_episode = episode >= warmup_episodes and episode % probe_episode_every == 0
        if is_probe_episode:
            epsilon = max(epsilon, 0.80)
            explore_noise_scale = np.maximum(explore_noise_scale, probe_noise_scale)

        env.max_delta_per_step = np.asarray(explore_noise_scale, dtype=np.float32)

        for step in range(max_steps):
            stability_noise_scale = np.maximum(min_noise, explore_noise_scale * max(0.15, 0.35 - 0.20 * progress))

            if episode < warmup_episodes:
                if best_episode_action is not None and np.random.rand() < 0.35:
                    action = best_episode_action.copy() + np.random.normal(loc=0.0, scale=warmup_noise, size=env.action_space.shape[0]).astype(np.float32)
                else:
                    action = np.random.uniform(env.low_act, env.high_act).astype(np.float32)
            else:
                sample = np.random.rand()
                if is_probe_episode:
                    probe_roll = np.random.rand()
                    if best_episode_action is not None and probe_roll < 0.60:
                        action = best_episode_action.copy() + np.random.normal(loc=0.0, scale=probe_noise_scale, size=env.action_space.shape[0]).astype(np.float32)
                    elif best_step_action is not None and probe_roll < 0.85:
                        action = best_step_action.copy() + np.random.normal(loc=0.0, scale=probe_noise_scale, size=env.action_space.shape[0]).astype(np.float32)
                    else:
                        action = np.random.uniform(env.low_act, env.high_act).astype(np.float32)
                elif best_episode_action is not None and sample < stability_replay_prob:
                    action = best_episode_action.copy() + np.random.normal(loc=0.0, scale=stability_noise_scale, size=env.action_space.shape[0]).astype(np.float32)
                elif sample < epsilon:
                    if np.random.rand() < full_range_prob or best_episode_action is None:
                        action = np.random.uniform(env.low_act, env.high_act).astype(np.float32)
                    elif best_step_action is not None and np.random.rand() < 0.50:
                        action = best_step_action.copy() + np.random.normal(loc=0.0, scale=explore_noise_scale, size=env.action_space.shape[0]).astype(np.float32)
                    else:
                        action = best_episode_action.copy() + np.random.normal(loc=0.0, scale=explore_noise_scale, size=env.action_space.shape[0]).astype(np.float32)
                else:
                    policy_action = np.asarray(agent.get_action(state), dtype=np.float32)
                    if best_episode_action is not None:
                        mix_ratio = 0.55 + 0.30 * progress
                        action = mix_ratio * best_episode_action + (1.0 - mix_ratio) * policy_action
                    elif best_step_action is not None:
                        action = 0.60 * best_step_action + 0.40 * policy_action
                    else:
                        action = policy_action
                    action = np.asarray(action, dtype=np.float32) + np.random.normal(loc=0.0, scale=policy_noise_scale, size=env.action_space.shape[0]).astype(np.float32)

            action = np.clip(np.asarray(action, dtype=np.float32), env.low_act, env.high_act)
            if not env.temperature_control_enabled:
                action[0] = env.nominal_action[0]
            if not env.boilup_control_enabled:
                action[1] = env.nominal_action[1]
            if not env.reflux_control_enabled:
                action[2] = env.nominal_action[2]

            next_state, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            transition_action = np.asarray(info.get('applied_action', info.get('safe_action', action)), dtype=np.float32)
            last_applied_action = transition_action.copy()

            if info.get('com_error'):
                total_com_errors += 1
                consecutive_com_errors += 1
                print(
                    f"Aspen COM issue on episode {episode} step {step}: {info.get('error_message', 'unknown error')} "
                    f"| action={info.get('safe_action')} | stage={info.get('failed_stage', 'unknown')} "
                    f"| total_errors={total_com_errors}"
                )
                agent.replay_buffer.push(state, transition_action, reward, next_state, True)
                episode_reward += reward
                if total_com_errors >= max_total_com_errors or consecutive_com_errors >= 3:
                    raise RuntimeError(
                        f"Aspen COM errors exceeded tolerance (total={total_com_errors}, consecutive={consecutive_com_errors}). "
                        f"Last issue: {info.get('error_message', 'unknown error')}"
                    )
                break
            else:
                consecutive_com_errors = 0

            if not info.get('converged', True):
                print(
                    f"Episode {episode} step {step} did not converge | restored={info.get('restored', False)} "
                    f"| action={info.get('safe_action')} | stage={info.get('failed_stage', 'unknown')}"
                )

            agent.replay_buffer.push(state, transition_action, reward, next_state, done)
            episode_reward += reward

            if info.get('converged', True) and reward > best_step_reward:
                best_step_reward = float(reward)
                best_step_action = transition_action.copy()

            if info.get('converged', True) and reward > episode_best_reward:
                episode_best_reward = float(reward)
                episode_best_action = transition_action.copy()

            if len(agent.replay_buffer) >= batch_size:
                for _ in range(updates_per_step):
                    update_metrics = agent.update(batch_size)
                    if update_metrics is not None:
                        update_metrics_window.append(update_metrics)

            state = next_state
            if done:
                break

        episode_rewards.append(episode_reward)
        episode_time = time.time() - start_time
        if best_step_action is not None:
            env.last_good_action = np.asarray(best_step_action, dtype=np.float32).copy()
        if episode_best_action is not None and episode_best_reward > best_episode_reward:
            best_episode_reward = float(episode_best_reward)
            best_episode_action = np.asarray(episode_best_action, dtype=np.float32).copy()

        action_str = re.sub(r'\s+', ',', str(last_applied_action.tolist()).strip())
        with open(csv_files['reward'], 'a', newline='') as file:
            csv.writer(file).writerow([episode, episode_reward])
        with open(csv_files['action'], 'a', newline='') as file:
            csv.writer(file).writerow([episode, action_str])
        with open(csv_files['state'], 'a', newline='') as file:
            csv.writer(file).writerow([episode, np.asarray(next_state).tolist()])
        with open(csv_files['time'], 'a', newline='') as file:
            csv.writer(file).writerow([episode, episode_time])

        avg_q1 = np.mean([m['q1_loss'] for m in update_metrics_window]) if update_metrics_window else np.nan
        avg_q2 = np.mean([m['q2_loss'] for m in update_metrics_window]) if update_metrics_window else np.nan
        valid_policy_losses = [m['policy_loss'] for m in update_metrics_window if not np.isnan(m['policy_loss'])]
        avg_policy = np.mean(valid_policy_losses) if valid_policy_losses else np.nan
        avg_alpha = np.mean([m['alpha'] for m in update_metrics_window]) if update_metrics_window else agent.alpha
        action_compact = '/'.join(f'{value:.4f}' for value in last_applied_action)
        print(
            f'Episode {episode} | Total Reward: {episode_reward:.4f} | Time: {episode_time:.2f}s '
            f'| Probe: {is_probe_episode} | Eps: {epsilon:.3f} '
            f'| Delta: {env.max_delta_per_step[0]:.2f}/{env.max_delta_per_step[1]:.3f}/{env.max_delta_per_step[2]:.3f} '
            f'| Act: {action_compact} '
            f'| BestStep: {best_step_reward:.4f} | BestEp: {best_episode_reward:.4f} | COM: {total_com_errors} '
            f'| AvgQ1: {avg_q1:.4f} | AvgQ2: {avg_q2:.4f} | AvgPolicy: {avg_policy:.4f} | Alpha: {avg_alpha:.4f}'
        )

    return episode_rewards


## 3. Connect Aspen Plus and Define RL Environment

In [3]:
import importlib
import sys

sys.modules.pop('CAPCodeLib', None)
import CAPCodeLib
importlib.reload(CAPCodeLib)
print('CAPCodeLib path:', CAPCodeLib.__file__)
print('Simulation has Run:', hasattr(CAPCodeLib.Simulation, 'Run'))
Simulation = CAPCodeLib.Simulation
assert hasattr(Simulation, 'Run'), 'Loaded CAPCodeLib.Simulation does not define Run()'

current_directory = Path.cwd()
aspen_filename = 'FT-PFR-3.27.apw'


SIM_VISIBLE = False  # set True only for manual Aspen debugging


def initialize_simulation(visible=SIM_VISIBLE):
    simulation = Simulation(
        AspenFileName=aspen_filename,
        WorkingDirectoryPath=str(current_directory),
        VISIBILITY=visible,
    )
    if hasattr(simulation, 'DialogSuppression'):
        simulation.DialogSuppression(True)
    if hasattr(simulation, 'DismissAspenDialogs'):
        simulation.DismissAspenDialogs()
    print(f'Aspen visible: {getattr(simulation.AspenSimulation, "Visible", "unknown")}')
    print(f'Aspen document: {simulation.abs_path}')
    return simulation


def create_simulation():
    return initialize_simulation(visible=SIM_VISIBLE)


sim = create_simulation()
try:
    ok = sim.Run()
except com_error as err:
    ok = False
    print(f'Initial Aspen COM error: {err}')
print('Run Done, Converge:', ok)


CAPCodeLib path: C:\Users\pcola\Desktop\캡스톤 강화학습 파일\레전드RL코드(제작중)\CAPCodeLib.py
Simulation has Run: True
아스펜 파일을 불러옵니다: C:\Users\pcola\Desktop\캡스톤 강화학습 파일\레전드RL코드(제작중)\FT-PFR-3.27.apw
>>> 아스펜 로드 완료! Visible=False
Aspen visible: False
Aspen document: C:\Users\pcola\Desktop\캡스톤 강화학습 파일\레전드RL코드(제작중)\FT-PFR-3.27.apw
Runtime = 0.33449482917785645
per_error value :  0
Run Done, Converge: True


In [4]:
def clamp01(x):
    return max(0.0, min(1.0, float(x)))


def normalize_utility_cost(cost, cost_ref=1000.0):
    cost = max(0.0, float(cost))
    cost_ref = max(1e-8, float(cost_ref))
    return np.log1p(cost) / np.log1p(cost_ref)


def compute_reward_v2(
    purity,
    selectivity,
    utility_cost,
    current_action,
    prev_action,
    w_purity=1.0,
    w_selectivity=1.0,
    w_balance=0.5,
    w_cost=0.7,
    w_action_penalty=0.0,
    utility_cost_ref=1000.0,
    purity_target=0.90,
    selectivity_target=0.30,
    target_steepness=10.0,
):
    purity = clamp01(purity)
    selectivity = clamp01(selectivity)

    quality_score = (w_purity * purity) + (w_selectivity * selectivity)
    balance_bonus = w_balance * np.sqrt(max(purity * selectivity, 0.0))

    purity_diff = purity - purity_target
    selectivity_diff = selectivity - selectivity_target
    target_score = 0.5 * (
        np.tanh(target_steepness * purity_diff) + np.tanh(target_steepness * selectivity_diff)
    )

    utility_penalty = w_cost * normalize_utility_cost(utility_cost, cost_ref=utility_cost_ref)

    action_penalty = 0.0
    if prev_action is not None:
        action_penalty = w_action_penalty * np.linalg.norm(current_action - prev_action)

    reward = quality_score + balance_bonus + target_score - utility_penalty - action_penalty

    reward_info = {
        'purity': purity,
        'selectivity': selectivity,
        'utility_cost': float(utility_cost),
        'quality_score': float(quality_score),
        'balance_bonus': float(balance_bonus),
        'target_score': float(target_score),
        'action_penalty': float(action_penalty),
        'utility_penalty': float(utility_penalty),
        'reward': float(reward),
    }
    return float(reward), reward_info



class AspenEnv(gym.Env):
    metadata = {'render_modes': []}

    def __init__(self, simulation, simulation_factory=None, max_steps=10):
        super().__init__()
        self.sim = simulation
        self.simulation_factory = simulation_factory
        self.low_act = np.array([220.0, 1.50, 0.30], dtype=np.float32)
        self.high_act = np.array([260.0, 2.20, 0.80], dtype=np.float32)

        self.action_space = gym.spaces.Box(low=self.low_act, high=self.high_act, dtype=np.float32)
        self.observation_space = gym.spaces.Box(
            low=np.array([0.0, 0.0, 0.0], dtype=np.float32),
            high=np.array([1.0, 1.0, 1.0], dtype=np.float32),
            dtype=np.float32,
        )

        self.target_stream = 'KERO'
        self.product_streams = ['KERO', 'DIESEL']
        self.max_steps = max_steps
        self.t = 0
        self.prev_action = None

        self.w_purity = 1.15
        self.w_selectivity = 1.10
        self.w_balance = 0.60
        self.w_cost = 0.45
        self.w_action_penalty = 0.00
        self.utility_cost_ref = 30.0
        self.purity_target = 0.925
        self.selectivity_target = 0.860
        self.target_steepness = 35.0
        self.default_nominal_action = np.array([240.0, 1.80, 0.50], dtype=np.float32)
        self.nominal_action = self._load_nominal_action_from_simulation()
        self.max_delta_per_step = np.array([2.5, 0.050, 0.050], dtype=np.float32)
        self.action_retry_sleep = 0.5
        self.rpc_restart_sleep = 2.0
        self._reset_control_flags()
        self.last_good_action = self.nominal_action.copy()
        self.state = np.zeros(3, dtype=np.float32)

    def _reset_control_flags(self):
        self.temperature_control_enabled = True
        self.boilup_control_enabled = True
        self.reflux_control_enabled = True

    def _is_rpc_error(self, err):
        text = str(err)
        return (
            'RPC ??? ??? ? ????' in text
            or 'The RPC server is unavailable' in text
            or '-2147023174' in text
        )

    def _restart_simulation(self, context, err):
        if self.simulation_factory is None:
            message = f'RPC recovery requested during {context}, but no simulation factory is configured: {err}'
            print(message)
            return False, message

        print(f'Restarting Aspen session after {context}: {err}')
        try:
            if self.sim is not None:
                try:
                    self.sim.CloseAspen()
                except Exception as close_err:
                    print(f'Ignoring Aspen close error during restart: {close_err}')
        except Exception:
            pass

        time.sleep(self.rpc_restart_sleep)

        try:
            replacement = self.simulation_factory()
            if hasattr(replacement, 'DialogSuppression'):
                replacement.DialogSuppression(True)
            run_ok = replacement.Run()
            if hasattr(replacement, 'DialogSuppression'):
                replacement.DialogSuppression(True)
            if not run_ok:
                last_run_error = getattr(replacement, 'last_run_error', None)
                raise RuntimeError(last_run_error or 'Aspen restart run did not converge.')

            self.sim = replacement
            self._reset_control_flags()
            self.nominal_action = self._load_nominal_action_from_simulation()
            self.last_good_action = self.nominal_action.copy()
            self.prev_action = self.nominal_action.copy()
            self.state = self._get_obs()
            print(f'Aspen session restarted successfully after {context}.')
            return True, None
        except Exception as restart_err:
            message = f'Aspen session restart failed after {context}: {restart_err}'
            print(message)
            return False, message

    def _get_process_metrics(self):
        try:
            purity = self.sim.get_kerosene_purity(self.target_stream, 8, 16)
            selectivity = self.sim.get_ft_selectivity(self.target_stream, self.product_streams, 8, 16)
            utility_cost = self.sim.get_utility_cost()
            return float(purity), float(selectivity), float(utility_cost)
        except Exception as err:
            raise RuntimeError(f'Process metric extraction failed: {err}') from err

    def _get_obs(self):
        purity, selectivity, utility_cost = self._get_process_metrics()
        normalized_cost = normalize_utility_cost(utility_cost, cost_ref=self.utility_cost_ref)
        return np.array([purity, selectivity, normalized_cost], dtype=np.float32)

    def _safe_float(self, value, fallback):
        try:
            return float(value)
        except (TypeError, ValueError):
            return float(fallback)

    def _load_nominal_action_from_simulation(self):
        nominal = self.default_nominal_action.copy()

        try:
            rplug_inputs = self.sim.BLK_RPLUG_GET_ME_ALL_INPUTS_BACK('R301')
            for key in ['Constant_Temp', 'ReactorTemperature', 'OutletTemp']:
                value = rplug_inputs.get(key)
                if value is not None:
                    nominal[0] = self._safe_float(value, nominal[0])
                    break
        except Exception as err:
            print(f'Could not read nominal reactor temperature from Aspen: {err}')

        try:
            d401_inputs = self.sim.BLK_RADFRAC_GET_ME_ALL_INPUTS_BACK('D401')
            nominal[1] = self._safe_float(d401_inputs.get('BoilupRatio'), nominal[1])
        except Exception as err:
            print(f'Could not read nominal D401 boilup ratio from Aspen: {err}')

        try:
            d401_rr_inputs = self.sim.BLK_RADFRAC_GET_ME_ALL_INPUTS_BACK('D401')
            nominal[2] = self._safe_float(d401_rr_inputs.get('Refluxratio'), nominal[2])
        except Exception as err:
            print(f'Could not read nominal D401 reflux ratio from Aspen: {err}')

        nominal = np.clip(nominal, self.low_act, self.high_act).astype(np.float32)
        print(f'Loaded nominal action from Aspen: {nominal.tolist()}')
        return nominal

    def _restore_stable_state(self):
        recovery_candidates = [('nominal', self.nominal_action.copy())]
        if self.last_good_action is not None:
            recovery_candidates.append(('last_good', self.last_good_action.copy()))

        for tag, candidate in recovery_candidates:
            try:
                candidate = self._sanitize_action(candidate)
                applied_action = self._apply_action(candidate)
                time.sleep(self.action_retry_sleep)
                converged = self.sim.Run()
                if hasattr(self.sim, 'DialogSuppression'):
                    self.sim.DialogSuppression(True)
                if not converged:
                    run_error = getattr(self.sim, 'last_run_error', None)
                    if run_error and self._is_rpc_error(run_error):
                        raise RuntimeError(run_error)
                    print(f'Restore attempt did not converge for {tag} action: {applied_action.tolist()}')
                    continue

                self.last_good_action = applied_action.copy()
                self.prev_action = applied_action.copy()
                print(f'Restored Aspen state using {tag} action: {applied_action.tolist()}')
                return True
            except Exception as err:
                if self._is_rpc_error(err):
                    raise
                print(f'Restore attempt failed for {tag} action: {err}')
        return False

    def _sanitize_action(self, action):
        action = np.asarray(action, dtype=np.float32)
        safe_action = np.clip(action, self.low_act, self.high_act).astype(np.float32)
        if not self.temperature_control_enabled:
            safe_action[0] = self.nominal_action[0]
        if not self.boilup_control_enabled:
            safe_action[1] = self.nominal_action[1]
        if not self.reflux_control_enabled:
            safe_action[2] = self.nominal_action[2]
        return safe_action

    def _apply_action(self, action):
        action = np.asarray(action, dtype=np.float32).copy()

        if self.temperature_control_enabled:
            try:
                self.sim.BLK_RPLUG_Set_T_SPEC_Constant_Temp('R301', float(action[0]))
            except Exception as err:
                if self._is_rpc_error(err):
                    raise RuntimeError(f'RPC error during temperature setter: {err}')
                print(f'Temperature control disabled for this session: {err}')
                self.temperature_control_enabled = False
                action[0] = self.nominal_action[0]

        if self.boilup_control_enabled:
            try:
                self.sim.BLK_RADFRAC_Set_BoilupRatio('D401', float(action[1]))
            except Exception as err:
                if self._is_rpc_error(err):
                    raise RuntimeError(f'RPC error during boilup setter: {err}')
                print(f'Boilup control disabled for this session: {err}')
                self.boilup_control_enabled = False
                action[1] = self.nominal_action[1]

        if self.reflux_control_enabled:
            try:
                self.sim.BLK_RADFRAC_Set_Refluxratio('D401', float(action[2]))
            except Exception as err:
                if self._is_rpc_error(err):
                    raise RuntimeError(f'RPC error during reflux setter: {err}')
                print(f'Reflux control disabled for this session: {err}')
                self.reflux_control_enabled = False
                action[2] = self.nominal_action[2]

        if not any([self.temperature_control_enabled, self.boilup_control_enabled, self.reflux_control_enabled]):
            raise RuntimeError('All manipulated variables are disabled for this Aspen session.')

        return action

    def _try_action(self, action, tag):
        try:
            action = self._apply_action(action)
            time.sleep(self.action_retry_sleep)
            converged = self.sim.Run()
            if hasattr(self.sim, 'DialogSuppression'):
                self.sim.DialogSuppression(True)
            run_error = getattr(self.sim, 'last_run_error', None)
            if not converged and run_error:
                return False, RuntimeError(run_error), f'{tag}/run', action
            return converged, None, f'{tag}/run-status', action
        except Exception as err:
            stage = f'{tag}/run' if self._is_rpc_error(err) else f'{tag}/setter'
            return False, err, stage, np.asarray(action, dtype=np.float32).copy()

    def step(self, action):
        requested_action = self._sanitize_action(action)
        fallback_actions = [('requested', requested_action)]

        if self.prev_action is not None:
            fallback_actions.append(('previous', self.prev_action.copy()))
        if self.last_good_action is not None:
            fallback_actions.append(('last_good', self.last_good_action.copy()))
        fallback_actions.append(('nominal', self.nominal_action.copy()))

        converged = False
        last_error = None
        action = requested_action
        failed_stage = None

        for tag, candidate_action in fallback_actions:
            candidate_action = self._sanitize_action(candidate_action)
            converged, step_error, failed_stage, applied_action = self._try_action(candidate_action, tag)
            if converged:
                action = applied_action
                if tag != 'requested':
                    print(f'Action fallback applied: {tag} -> {applied_action.tolist()}')
                break
            if step_error is not None:
                last_error = step_error
                print(f'Step error at {tag} action: {step_error}')
            else:
                print(f'Non-converged at {tag} action: {applied_action.tolist()}')
            time.sleep(self.action_retry_sleep)

        if last_error is not None and not converged:
            restored = False
            restart_error = None
            if self._is_rpc_error(last_error):
                restored, restart_error = self._restart_simulation(failed_stage or 'step', last_error)
            else:
                try:
                    restored = self._restore_stable_state()
                except Exception as restore_err:
                    if self._is_rpc_error(restore_err):
                        last_error = restore_err
                        restored, restart_error = self._restart_simulation('stable-state restore', restore_err)
                    else:
                        print(f'Restore attempt failed after action error: {restore_err}')

            self.state = np.zeros(3, dtype=np.float32)
            reward = -50.0
            terminated = True
            truncated = False
            error_message = str(last_error)
            if restart_error:
                error_message = f'{error_message} | restart={restart_error}'
            info = {
                'converged': False,
                'com_error': True,
                'error_message': error_message,
                'safe_action': requested_action.tolist(),
                'failed_stage': failed_stage,
                'restored': restored,
                'rpc_restarted': bool(restored and self._is_rpc_error(last_error)),
            }
            return self.state, reward, terminated, truncated, info

        if not converged:
            try:
                restored = self._restore_stable_state()
                self.state = np.zeros(3, dtype=np.float32)
                reward = -50.0
                terminated = True
                truncated = False
                info = {'converged': False, 'com_error': False, 'safe_action': requested_action.tolist(), 'failed_stage': failed_stage, 'restored': restored}
                return self.state, reward, terminated, truncated, info
            except Exception as restore_err:
                if self._is_rpc_error(restore_err):
                    restored, restart_error = self._restart_simulation('non-converged restore', restore_err)
                    self.state = np.zeros(3, dtype=np.float32)
                    reward = -50.0
                    terminated = True
                    truncated = False
                    error_message = str(restore_err)
                    if restart_error:
                        error_message = f'{error_message} | restart={restart_error}'
                    info = {
                        'converged': False,
                        'com_error': True,
                        'error_message': error_message,
                        'safe_action': requested_action.tolist(),
                        'failed_stage': failed_stage,
                        'restored': restored,
                        'rpc_restarted': restored,
                    }
                    return self.state, reward, terminated, truncated, info
                raise

        self.last_good_action = action.copy()
        try:
            purity, selectivity, utility_cost = self._get_process_metrics()
        except Exception as err:
            restored = False
            restart_error = None
            if self._is_rpc_error(err):
                restored, restart_error = self._restart_simulation('metric extraction', err)
            else:
                try:
                    restored = self._restore_stable_state()
                except Exception as restore_err:
                    if self._is_rpc_error(restore_err):
                        restored, restart_error = self._restart_simulation('metric extraction restore', restore_err)
                    else:
                        print(f'Restore attempt failed after metric extraction error: {restore_err}')

            self.state = np.zeros(3, dtype=np.float32)
            reward = -50.0
            terminated = True
            truncated = False
            error_message = str(err)
            if restart_error:
                error_message = f'{error_message} | restart={restart_error}'
            info = {
                'converged': False,
                'com_error': True,
                'error_message': error_message,
                'safe_action': action.tolist(),
                'failed_stage': 'metrics',
                'restored': restored,
                'rpc_restarted': bool(restored and self._is_rpc_error(err)),
            }
            return self.state, reward, terminated, truncated, info

        self.state = np.array([
            purity,
            selectivity,
            normalize_utility_cost(utility_cost, cost_ref=self.utility_cost_ref),
        ], dtype=np.float32)
        reward, reward_info = compute_reward_v2(
            purity=purity,
            selectivity=selectivity,
            utility_cost=utility_cost,
            current_action=action,
            prev_action=self.prev_action,
            w_purity=self.w_purity,
            w_selectivity=self.w_selectivity,
            w_balance=self.w_balance,
            w_cost=self.w_cost,
            w_action_penalty=self.w_action_penalty,
            utility_cost_ref=self.utility_cost_ref,
            purity_target=self.purity_target,
            selectivity_target=self.selectivity_target,
            target_steepness=self.target_steepness,
        )

        self.prev_action = action.copy()
        self.t += 1
        terminated = self.t >= self.max_steps
        truncated = False
        info = {'converged': True, 'applied_action': action.tolist(), **reward_info}

        print(
            f"Step {self.t}: Purity={reward_info['purity']:.4f}, "
            f"Selectivity={reward_info['selectivity']:.4f}, "
            f"Util_Cost={reward_info['utility_cost']:.4f}, "
            f"Reward={reward_info['reward']:.4f}"
        )

        return self.state, reward, terminated, truncated, info

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.t = 0
        self.prev_action = self.nominal_action.copy()
        try:
            restored = self._restore_stable_state()
            self.state = self._get_obs()
            return self.state, {'converged': restored, 'com_error': False}
        except Exception as err:
            if self._is_rpc_error(err):
                restarted, restart_error = self._restart_simulation('reset', err)
                if restarted:
                    return self.state, {'converged': True, 'com_error': False, 'restarted': True}
                message = restart_error or str(err)
            else:
                message = str(err)
            print(f'Reset error: {message}')
            self.state = np.zeros(3, dtype=np.float32)
            return self.state, {'converged': False, 'com_error': True, 'error_message': message}

    def render(self):
        pass


## 4. Run SAC Training

In [ ]:
env = AspenEnv(simulation=sim, simulation_factory=create_simulation, max_steps=1)

gamma = 0.0
tau = 0.02
alpha = 0.15
a_lr = 5e-5
q_lr = 5e-4
p_lr = 5e-4
buffer_maxlen = 1_000_000

max_episodes = 500
max_steps = env.max_steps
batch_size = 16

expt_code = '_expt206_static_global_sac'
run = 10
run_seed = {1: 212, 2: 123, 3: 456, 4: 9, 5: 11, 6: 88, 7: 101, 8: 1122, 9: 2002, 10: 5555}
seed = run_seed[run]

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

agent = SACAgent(env, gamma, tau, alpha, q_lr, p_lr, a_lr, buffer_maxlen)


def validate_aspen_session(env):
    state, info = env.reset()
    if info.get('com_error'):
        raise RuntimeError(f"Aspen reset failed: {info.get('error_message', 'unknown error')}")
    return state, info


def do_the_thing(experiment_max_num):
    validate_aspen_session(env)
    results = []
    for num in range(experiment_max_num):
        filename = f'{expt_code}_500iter_RNG{seed}_{run}_{num}'
        rewards = mini_batch_train(env, agent, max_episodes, max_steps, batch_size, filename)
        results.append(rewards)
    print('Done with runs.')
    return results


experiment_max_num = 1
training_results = do_the_thing(experiment_max_num)


Could not read nominal reactor temperature from Aspen: 'NoneType' object has no attribute 'Value'
Could not read nominal D401 boilup ratio from Aspen: 'NoneType' object has no attribute 'Value'
Could not read nominal D401 reflux ratio from Aspen: 'NoneType' object has no attribute 'Value'
Loaded nominal action from Aspen: [240.0, 1.7999999523162842, 0.5]
Runtime = 5.571203947067261
per_error value :  1
Runtime = 0.24632477760314941
per_error value :  1
Aspen PER_ERROR remained non-zero after retry.
Restore attempt did not converge for last_good action: [240.0, 1.7999999523162842, 0.5]
Runtime = 0.23878073692321777
per_error value :  1
Runtime = 0.3370530605316162
per_error value :  1
Aspen PER_ERROR remained non-zero after retry.
Restore attempt did not converge for nominal action: [240.0, 1.7999999523162842, 0.5]
Runtime = 0.2355482578277588
per_error value :  1
Runtime = 0.2613661289215088
per_error value :  1
Aspen PER_ERROR remained non-zero after retry.
Restore attempt did not con

## 5. Visualize Results

In [4]:
from pathlib import Path
import pandas as pd
import plotly.express as px

report_dir = Path.cwd()
print(f'Looking for reward CSV files in: {report_dir}')
reward_csvs = sorted(report_dir.glob('report_rewards_*.csv'))

if reward_csvs:
    reward_df = pd.concat([pd.read_csv(csv_path).assign(file=csv_path.name) for csv_path in reward_csvs], ignore_index=True)
    fig = px.line(reward_df, x='episode', y='total_reward', color='file', title='Episode Reward History')
    fig.show()
else:
    print('No reward CSV files were found in the current directory.')


Looking for reward CSV files in: C:\Users\pcola\Desktop\캡스톤 강화학습 파일\레전드RL코드(제작중)


## 6. Close Aspen Plus Session

In [ ]:
try:
    sim.CloseAspen()
except Exception as exc:
    print(f'Aspen close skipped: {exc}')
